# SepsisGuard — GRPO Training Notebook

Train 4 multi-agent roles (Nurse, Lab, Pharmacist, Physician) using TRL GRPO with Unsloth 4-bit quantization.

This notebook:
1. Loads a quantized Qwen 2.5-3B model
2. Connects to the live SepsisGuard environment
3. Collects initial rollouts for the prompt dataset
4. Runs GRPO training with online environment rewards
5. Evaluates and plots reward improvement vs heuristic baseline

In [ ]:
!pip install -q -U "unsloth[colab-new]" openenv-core "trl>=0.12" vllm datasets matplotlib
!pip install -q requests httpx

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
FastLanguageModel.for_inference(model)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
import os, requests, uuid

ENV_URL = os.environ.get("ENV_BASE_URL", "https://YOUR-USERNAME-sepsisguard.hf.space")

class EnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def reset(self, task_name, seed, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/reset",
                          json={"task_name": task_name, "seed": seed},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def step(self, actions, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/step",
                          json={"actions": actions},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def create_session(self):
        r = requests.post(f"{self.base_url}/session", timeout=10)
        r.raise_for_status()
        return r.json()["session_id"]

    def delete_session(self, session_id):
        try:
            requests.delete(f"{self.base_url}/session/{session_id}", timeout=5)
        except Exception:
            pass

env = EnvClient(ENV_URL)
info = env.reset(task_name="task1_textbook", seed=42)
print(f"Connected to {ENV_URL}")
print(f"Tick: {info['info']['tick']}, Roles: {list(info['observations'].keys())}")

In [ ]:
import sys
sys.path.insert(0, "/content/sepsisguard")
from training.rollout_collector import collect_rollouts

N_EPISODES = 4
TASK = "task1_textbook"

rollouts = collect_rollouts(model, tokenizer, env, n_episodes=N_EPISODES, task=TASK)
print(f"Collected {len(rollouts)} rollout steps across {N_EPISODES} episodes")
print(f"Roles in rollouts: {set(r['role'] for r in rollouts)}")
print(f"Sample prompt length: {len(rollouts[0]['prompt'])} chars")
print(f"Sample completion: {rollouts[0]['completion'][:200]}")

In [ ]:
import torch
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from training.reward_shaping import make_online_sepsis_reward_fn, format_reward_fn

FastLanguageModel.for_training(model)

train_dataset = Dataset.from_list([
    {"prompt": r["prompt"]} for r in rollouts
])

cfg = GRPOConfig(
    output_dir="./sepsis-grpo",
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=200,
    learning_rate=2e-5,
    warmup_steps=20,
    logging_steps=5,
    save_steps=50,
    max_prompt_length=3000,
    max_completion_length=128,
    bf16=True,
    report_to="none",
)

reward_fn_env = make_online_sepsis_reward_fn(
    env_url=ENV_URL,
    task_name=TASK,
    seed=42,
    max_steps_per_eval=8,
)

reward_log = {"steps": [], "env_reward": [], "format_reward": []}

class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "reward" in str(logs):
            reward_log["steps"].append(state.global_step)
            reward_log["env_reward"].append(logs.get("rewards/reward_func_0", 0.0))
            reward_log["format_reward"].append(logs.get("rewards/reward_func_1", 0.0))
        if state.global_step % 50 == 0 and state.global_step > 0:
            sample_prompt = train_dataset[0]["prompt"]
            inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)
            model.eval()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            model.train()
            generated = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            print(f"\n=== Step {state.global_step} sample ===")
            print(generated[:300])
            print("=" * 40)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn_env, format_reward_fn],
    args=cfg,
    train_dataset=train_dataset,
    callbacks=[RewardLogger()],
)

print(f"Training config: {cfg.max_steps} steps, lr={cfg.learning_rate}, "
      f"batch={cfg.per_device_train_batch_size}x{cfg.gradient_accumulation_steps}, "
      f"generations={cfg.num_generations}")
print("Starting GRPO training...")
trainer.train()
print("Training complete.")

In [ ]:
model.save_pretrained("./sepsis-grpo-lora")
tokenizer.save_pretrained("./sepsis-grpo-lora")

model.save_pretrained_merged(
    "./sepsis-grpo-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("LoRA adapters saved to ./sepsis-grpo-lora")
print("Merged model saved to ./sepsis-grpo-merged")

In [ ]:
import requests, json, re
from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician
from training.prompts import build_role_prompt

def run_episode(env_client, task_name, seed, agent_fn, session_id=None):
    """Run one episode. agent_fn(role, obs) -> action dict."""
    bundle = env_client.reset(task_name=task_name, seed=seed, session_id=session_id)
    done = False
    step_rewards = []
    while not done:
        obs = bundle["observations"]
        actions = {}
        for role in ("nurse", "lab", "pharmacist", "physician"):
            actions[role] = agent_fn(role, obs[role])
        bundle = env_client.step(actions, session_id=session_id)
        step_rewards.append(bundle.get("team_reward", 0.0))
        done = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": session_id} if session_id else {},
                          timeout=30).json()
    return grader.get("score", 0.0), step_rewards

nurse_h, lab_h, pharma_h, phys_h = HeuristicNurse(), HeuristicLab(), HeuristicPharmacist(), HeuristicPhysician()
heuristic_agents = {"nurse": nurse_h, "lab": lab_h, "pharmacist": pharma_h, "physician": phys_h}

def heuristic_agent_fn(role, obs):
    return heuristic_agents[role].decide(obs)

def make_llm_agent_fn(model, tokenizer, target_role):
    def agent_fn(role, obs):
        if role != target_role:
            return heuristic_agents[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with __import__("torch").no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and "operation" in parsed:
                return parsed
        except Exception:
            pass
        m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
        return heuristic_agents[role].decide(obs)
    return agent_fn

N_EVAL = 5
EVAL_TASKS = ["task1_textbook"]

print("=" * 60)
print("EVALUATION: Heuristic Baseline vs Trained Model")
print("=" * 60)

baseline_scores = {}
for task in EVAL_TASKS:
    scores = []
    for ep in range(N_EVAL):
        sid = env.create_session()
        score, _ = run_episode(env, task, seed=100 + ep, agent_fn=heuristic_agent_fn, session_id=sid)
        scores.append(score)
        env.delete_session(sid)
    baseline_scores[task] = scores
    print(f"\nHeuristic [{task}]: mean={sum(scores)/len(scores):.4f}  scores={[round(s,3) for s in scores]}")

FastLanguageModel.for_inference(model)
trained_scores = {}
for task in EVAL_TASKS:
    for target_role in ("nurse", "lab", "pharmacist", "physician"):
        llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
        scores = []
        for ep in range(N_EVAL):
            sid = env.create_session()
            score, _ = run_episode(env, task, seed=100 + ep, agent_fn=llm_fn, session_id=sid)
            scores.append(score)
            env.delete_session(sid)
        key = f"{task}_{target_role}"
        trained_scores[key] = scores
        mean_s = sum(scores) / len(scores)
        base_mean = sum(baseline_scores[task]) / len(baseline_scores[task])
        delta = mean_s - base_mean
        print(f"Trained [{task}/{target_role}]: mean={mean_s:.4f}  delta={delta:+.4f}  scores={[round(s,3) for s in scores]}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if reward_log["steps"]:
    axes[0].plot(reward_log["steps"], reward_log["env_reward"], label="Env Reward", marker="o", markersize=3)
    axes[0].plot(reward_log["steps"], reward_log["format_reward"], label="Format Reward", marker="s", markersize=3)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Reward")
    axes[0].set_title("GRPO Training Reward Curves")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No reward logs captured\n(check TRL logging keys)",
                 ha="center", va="center", transform=axes[0].transAxes)
    axes[0].set_title("GRPO Training Reward Curves")

task = EVAL_TASKS[0]
roles = ["nurse", "lab", "pharmacist", "physician"]
base_mean = sum(baseline_scores[task]) / len(baseline_scores[task])
trained_means = []
for role in roles:
    key = f"{task}_{role}"
    trained_means.append(sum(trained_scores[key]) / len(trained_scores[key]))

x = range(len(roles) + 1)
labels = ["Heuristic"] + [r.capitalize() for r in roles]
values = [base_mean] + trained_means
colors = ["#888888"] + ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
bars = axes[1].bar(x, values, color=colors)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=15)
axes[1].set_ylabel("Score")
axes[1].set_title(f"Heuristic vs Trained ({task})")
axes[1].set_ylim(0, 1.0)
axes[1].axhline(y=base_mean, color="#888888", linestyle="--", alpha=0.5)
axes[1].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to training_results.png")